# Human + Multi-Agent Organizations

## Scenario: Northstar Commerce needs a cross-functional incident investigation—not an autonomous org chart

EU checkout conversion has fallen 31%. A human sponsor asks for a diagnosis, business impact, and a reviewer-ready mitigation proposal. The human retains authority over goals, data access, customer commitments, and any production action.

**Outcomes:** create bounded digital-worker role contracts; delegate typed work orders; supervise before/during/after a run; join reviewable artifacts; test when a team improves a single-agent baseline; and keep human authority explicit.

**Run:** Default cells are deterministic and credential-free. They simulate operating contracts, not real production actions.

![Human + multi-agent organization](../../../assets/human-multi-agent-organization.svg)

The manager coordinates bounded work; it does not inherit the human's authority. Specialists receive least-privilege sources and return artifacts containing evidence, confidence, and limitations. A human approver decides the consequential next step.

## 1. Organization design starts with accountability

A useful mixed team has four layers: human purpose and authority; a manager that translates the goal into bounded work; specialists that create typed, attributable artifacts; and human review of consequences. Delegation is not abdication. The manager should surface conflicts and uncertainty, not silently turn them into a decision.

Work orders define scope, approved sources/tools, expected artifact, budget, deadline, no-go actions, and escalation. Artifacts carry evidence IDs, confidence, and limitations so downstream agents and humans can audit them.

In [1]:
from lab import OrganizationRun, delegate, produce_artifacts, human_review, Status

run = OrganizationRun('northstar-acme', 'Investigate EU checkout conversion; prepare but do not execute mitigation.')
delegate(run)
for order in run.work:
    print(order.id, '|', order.owner, '| sources:', order.allowed_sources, '| write:', order.write_access)
assert all(not order.write_access for order in run.work)

research | research-agent | sources: ('runbooks', 'incidents') | write: False
data | data-agent | sources: ('metrics', 'customers') | write: False
code | coding-agent | sources: ('deployments', 'repository') | write: False
analysis | analysis-agent | sources: () | write: False
review | review-agent | sources: () | write: False


## 2. Delegation and context isolation

The research agent can see runbooks/incidents, the data agent metrics/customer segments, and the coding agent deployment/repository data. The analysis and review roles receive artifacts rather than every raw transcript. This separation reduces unnecessary exposure, makes provenance visible, and prevents one specialist from accidentally using another's unrestricted permissions.

A team is not inherently better. For a predictable status report, the manager and five specialists add coordination cost without new value. Use the organization only if independently scoped work improves the release metric enough to justify extra latency, cost, and operational complexity.

In [2]:
produce_artifacts(run)
for artifact in run.artifacts:
    print(f'[{artifact.owner}] {artifact.claim}')
    print('  evidence:', artifact.evidence_ids, 'confidence:', artifact.confidence)
    print('  limits:', artifact.limits)
assert run.status is Status.REVIEW
assert all(artifact.evidence_ids for artifact in run.artifacts)

[research] Provider region-mismatch runbook matches symptoms.
  evidence: ['runbook:payment-region-v3', 'incident:418'] confidence: 0.82
  limits: ['Runbook is advisory, not authorization.']
[data] EU conversion is down 31%; Gold-tier cohort has 17 affected accounts.
  evidence: ['metric:eu-conversion', 'customer:gold-cohort'] confidence: 0.95
  limits: ['Correlation does not prove deployment cause.']
[code] 08:42 deployment changed EU provider-region mapping.
  evidence: ['deployment:842', 'diff:provider-map'] confidence: 0.91
  limits: ['No live configuration was inspected.']
[analysis] Most likely cause: region mapping regression; prepare scoped rollback proposal.
  evidence: ['artifact:research', 'artifact:data', 'artifact:code'] confidence: 0.84
  limits: ['Requires reviewer validation before action.']
[review] Evidence supports a proposal, not execution; confirm change window and exact rollback scope.
  evidence: ['artifact:analysis'] confidence: 0.9
  limits: ['Human approver mu

## 3. Supervision before, during, and after

**Before:** human sponsor defines priority, purpose, authority, SLO, tool/data boundaries, review criteria, and escalation triggers.

**During:** the manager reports work-order state, spend, evidence coverage, source freshness, conflicts, and replan reason. Set caps on delegation depth, messages, per-agent turns, tool calls, time, and cost.

**After:** the human reviews material artifacts and the exact proposed action. Trace and evaluation results feed a regression set. Watch review burden: if people repeatedly repair an agent's output, the organization is not yet reliable enough to scale.

## 4. Deliberate failure: a proposal is not permission

The review agent finds sufficient evidence for a *proposal*, but requests confirmation of the change window and rollback scope. The correct default is escalation. A model recommendation, an internal handoff, or an agent-manager instruction is never a server-side authorization decision.

In [3]:
human_review(run, approve=False)
print('status:', run.status.value)
print('events:', run.events)
assert run.status is Status.ESCALATED
assert run.approved is False

# A new run—or an authenticated, scoped approval for the exact action—would be required before execution.
# The lab intentionally has no production executor.

status: escalated
events: ['manager:delegated-5-bounded-work-orders', 'manager:assembled-review-packet', 'human:escalated-for-more-evidence']


## 5. Evaluate the organization, not only its final report

Measure: task quality and evidence support; policy and scope adherence; useful division of labor; duplicate work; manager routing quality; human escalation/rework; latency/cost; and safety outcomes. Run the same incident with a bounded single agent and the organization. If the team does not measurably improve a target metric, keep the simpler baseline.

Also test collective failure modes: correlated hallucinations, reciprocal delegation loops, agent-to-agent prompt injection, hidden state sharing, role confusion, scope creep, manager overreach, and review fatigue.

## Exercises

1. Add a customer-communications digital worker that can only draft—not send—an update. Include a reviewer and approval fingerprint.
2. Make the research and data artifacts conflict. Implement a manager policy that escalates rather than averages their confidence.
3. Add per-role cost budgets and show a manager cancelling a nonessential research task.
4. Write a single-agent baseline for a simple status report and explain why the organization should not be used.
5. Define a dashboard that lets the human oversee purpose, scope, status, evidence coverage, risk, cost, owner, and escalation.

## References

- [OpenAI practical guide to building agents](https://openai.com/business/guides-and-resources/a-practical-guide-to-building-ai-agents/)
- [Anthropic: AI organizations can be more effective but less aligned](https://alignment.anthropic.com/2026/ai-organizations/)
- [Manager agent research challenge](https://arxiv.org/abs/2510.02557)
- [Human oversight of agentic systems in practice](https://arxiv.org/abs/2606.05391)
- [NIST AI RMF](https://www.nist.gov/itl/ai-risk-management-framework)